In [2]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests


In [3]:
from langchain.agents import create_agent

In [4]:
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["WEATHERSTACK_API_KEY"] = os.getenv("WEATHERSTACK_API_KEY")

In [5]:
search_tool = TavilySearch(max_results=3) 

In [6]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [7]:
result = search_tool.invoke("what is the leet code problem 1?")
result

{'query': 'what is the leet code problem 1?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://dev.to/savviesammie/two-sum-leetcode-problem-1-solution-in-python-4al9',
   'title': 'Two Sum - LeetCode problem 1 solution in python - DEV Community',
   'content': "Two sum problems: Link to problem on leetCode\n\nImage description  \n My First Approach The simple approach is to use two nested loops. when we have two nested loops, one is the outer loop while the other is the inner loop. Since the first input is a list, the outer loop iterates over all the elements of the list, and the inner loop iterates from the current index of the outer loop up to the end of the list. here's the part where you want to grab a pen and paper and write a simple algorithm [...] ⏩ Writing the Code\n\n✅ ✅ ✅ The code works\n\nwhile the solution passed all the cases, You can see that the time complexity is not good enough and the code can be improved to execute faster. h

In [8]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    api_key = os.environ["GROQ_API_KEY"]
) 

In [9]:
response = llm.invoke("what month is it?")
response

AIMessage(content='It’s September 2026.', additional_kwargs={'reasoning_content': 'User asks: "what month is it?" They are likely referring to current date. The system says current date is 2026-09-11. So it\'s September. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 76, 'total_tokens': 131, 'completion_time': 0.062666079, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.004494641, 'prompt_tokens_details': None, 'queue_time': 0.306687608, 'total_time': 0.06716072}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c9afb2bdb4', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08ff7-a2ea-7422-84d2-163d60c55f3a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 55, 'total_tokens': 131, 'output_token_details': {'reasoning': 39}})

In [10]:
tools = [search_tool,get_weather]

In [11]:
agent = create_agent(
    model = llm,
    tools = tools 
)

In [12]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the current weather in Ongole"}
    ]
})
response

{'messages': [HumanMessage(content='Find the current weather in Ongole', additional_kwargs={}, response_metadata={}, id='644a0b60-c9cd-40b7-abda-73fcb8314e43'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use get_weather function.', 'tool_calls': [{'id': 'fc_b93af116-ce1b-409a-9449-677c0f6a4cce', 'function': {'arguments': '{"city":"Ongole"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 343, 'total_tokens': 377, 'completion_time': 0.037202078, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.02092199, 'prompt_tokens_details': None, 'queue_time': 0.344901649, 'total_time': 0.058124068}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1c0f5282a2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08ff7-a5eb-7f41-9b0c-5ccfdfa473ba-0', tool_calls=[{'name': 'get_weather', 'args

In [13]:
print(response["messages"][-1].content)

**Current Weather in Ongole**

- **Temperature:** 35 °C  
- **Condition:** Overcast  
- **Humidity:** 38%  

Feel free to let me know if you’d like additional details (e.g., wind speed, forecast, or a different location)!


In [14]:
for msg in response["messages"]:
    print(msg.content)

Find the current weather in Ongole

City: Ongole
Temperature: 35°C
Weather: Overcast 
Humidity: 38%
**Current Weather in Ongole**

- **Temperature:** 35 °C  
- **Condition:** Overcast  
- **Humidity:** 38%  

Feel free to let me know if you’d like additional details (e.g., wind speed, forecast, or a different location)!
